# Place SNP Orders

In [1]:
## THIS CELL SHOULD BE IN ALL VSCODE NOTEBOOKS ##

MARKET = "SNP"

# Set the root
from from_root import from_root # type: ignore
ROOT = from_root()

import pandas as pd # type: ignore
from loguru import logger # type: ignore

pd.options.display.max_columns = None
pd.set_option('display.precision', 2)

from pathlib import Path
import sys

# Add `src` and ROOT to _src.pth in .venv to allow imports in VS Code
from sysconfig import get_path

if "src" not in Path.cwd().parts:
    src_path = str(Path(get_path("purelib")) / "_src.pth")
    with open(src_path, "w") as f:
        f.write(str(ROOT / "src\n"))
        f.write(str(ROOT))
        if str(ROOT) not in sys.path:
            sys.path.insert(1, str(ROOT))

# Start the Jupyter loop
from ib_async import util # type: ignore

util.startLoop()

logger.add(sink=ROOT / "log" / "ztest.log", mode="w")

1

## Imports


In [2]:
from datetime import datetime

from ib_async import IB

from ibfuncs import get_open_orders, make_ib_orders, place_orders, quick_pf
from utils import (arrange_orders, get_pickle, how_many_days_old, load_config,
                   pickle_me, strip_split, yes_or_no)

## Set constants

In [3]:
config = load_config(MARKET)
port = config.get('PORT')
MARGINPERORDER = config.get('MARGINPERORDER')

# Get orders to be placed

In [4]:
# Check age of pickles

nakeds_path = ROOT / 'data' / 'snp_nakeds.pkl'
txt = f'snp_nakeds.pkl is {how_many_days_old(nakeds_path): 0.2f} days old. Want to load??'
ans = yes_or_no(txt)

if ans:
    df_opts = get_pickle(nakeds_path)
    print('\n\n')

    cols = strip_split('ib_symbol,strike,safe_strike,right,dte,bsPrice, price, xPrice, margin, rom')
    print(df_opts[cols].head())
else:
    print('Bye!!!')

snp_nakeds.pkl is  0.06 days old. Want to load?? (y/n):  y





  ib_symbol  strike  safe_strike right   dte  bsPrice  price  xPrice  margin  \
0       VST    72.0           72     P  0.46     0.04   0.20    0.20  3079.2   
1     CMCSA    40.0           39     P  0.46     0.27   1.41    2.41  1734.0   
2      DLTR    68.0           65     P  0.46     0.56   4.62    8.06  3199.6   
3       MRK   119.0          116     P  0.46     0.85   3.27    6.27  5029.6   
4       HRL    32.0           31     P  0.46     0.18   1.06    2.06  1386.0   

      rom  
0    5.15  
1  110.19  
2  199.72  
3   98.84  
4  117.84  


# Check Open orders

In [5]:
# check open orders
with IB().connect(port=port, clientId=10) as ib:
    dfo = get_open_orders(ib)
    dfp = quick_pf(ib)

if not dfo.empty:
    remove_opens = set(dfo.symbol.to_list())
else:
    remove_opens = set()

# make a list of symbols to be removed from df_opts
if not dfp.empty:
    remove_positions = set(dfp.symbol.to_list())
else:
    remove_positions = set()

remove_ib_syms = remove_opens | remove_positions

# get the target options to plant
dft = df_opts[~df_opts.ib_symbol.isin(remove_ib_syms)].reset_index(drop=True)

print(f'{len(dft)} open orders found\n\n')

dft[cols].head()

2240 open orders found




,ib_symbol,strike,safe_strike,right,dte,bsPrice,price,xPrice,margin,rom
0,VST,72.0,72,P,0.46,0.04,0.20,0.20,3079.2,5.15
1,CMCSA,40.0,39,P,0.46,0.27,1.41,2.41,1734.0,110.19
2,MRK,119.0,116,P,0.46,0.85,3.27,6.27,5029.6,98.84
3,HRL,32.0,31,P,0.46,0.18,1.06,2.06,1386.0,117.84
4,HRL,32.0,31,P,0.46,0.18,1.06,2.06,1386.0,117.84


# Arrange and make orders

In [6]:
df_nakeds = arrange_orders(dft, maxmargin=MARGINPERORDER)
cos = make_ib_orders(df_nakeds)

# PLACE THE ORDER